In [19]:
import google.generativeai as genai
from dotenv import load_dotenv
# API KEY 정보 로드
load_dotenv()


# 내 키로 접근 가능한 모델 중 '임베딩' 기능이 있는 것만 출력
for m in genai.list_models():
    if 'embedContent' in m.supported_generation_methods:
        print(f"사용 가능한 모델명: {m.name}")

C:\workspace\class_rag\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\junsu\AppData\Local\Temp\ipykernel_17556\1687564974.py:11: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


사용 가능한 모델명: models/gemini-embedding-001


In [36]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_core.prompts import PromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_google_genai import GoogleGenerativeAIEmbeddings

from dotenv import load_dotenv
import os


# API KEY 정보 로드
load_dotenv()


# LLM 엔진 (대답하는 기계)
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")

# 답변 한번에 나옴
# reponse = llm.invoke("안녕? 대한민국의 수도는 어디냐?")
# print(reponse.content)

# 스트리밍 출력
# for chunk in llm.stream("Java와 Python의 차이점을 짧게 알려줘."):
#     print(chunk.content, end="", flush=True)


# 1. 문서 로드 (PyMuPDFLoader 사용)
loader = PyMuPDFLoader("./files/profile_test.pdf") # 프로젝트 폴더에 있는 PDF 파일명
docs = loader.load()

# print(docs)

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,     # 한 조각당 최대 글자 수
    chunk_overlap=50,   # 조각 간 겹치는 글자 수 (문맥 유실 방지용) chunk size에 10%~20%
    length_function=len,
    is_separator_regex=False
)

# 로드된 문서 분할
# loader.load()로 가져온 docs 객체 를 넣는다.
split_docs = text_splitter.split_documents(docs)

print(f"원본 문서 페이지수 : {len(docs)}")
print(f"분할된 조각(Chunk) 수 : {len(split_docs)}")


# 2. 임베딩 모델 설정 (텍스트를 숫자로 바꾸는 엔진)
embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")


# 3. 벡터 DB 생성 (FAISS 사용)
# docs를 숫자로 바꿔서 메모리 DB(FAISS)에 저장합니다.
# vectorstore = FAISS.from_documents(docs, embeddings)
vectorstore = FAISS.from_documents(docs, embeddings)

print(vectorstore)


# 4. 검색기(Retriever) 설정
retriever = vectorstore.as_retriever()

원본 문서 페이지수 : 5
분할된 조각(Chunk) 수 : 8


In [ ]:

# retriever = vectorstore.as_retriever()
# # 검색된 문서 조각들을 직접 확인해보기
# question = "문서에 지원자 이름이 뭐야?"
# relevant_docs = retriever.invoke(question)
#
# print(f"--- 검색된 문서 조각 개수: {len(relevant_docs)} ---")
# for i, doc in enumerate(relevant_docs):
#     print(f"[{i}] 내용: {doc.page_content[:100]}...")
#     print(f"[{i}] 출처: {doc.metadata}\n")

In [41]:
# 내부에서 일어나는 일을 수동으로 해보기
question = "지원자 이름이 뭐야?"

# 1. 질문을 임베딩해서 숫자로 바꾸기 (내부 로직)
query_vector = embeddings.embed_query(question)
print(f"질문 벡터 샘플(5개): {query_vector[:5]}")
# 출력: [0.012, -0.045, 0.231, ...] 이런 식으로 나옵니다.

# 2. 이 숫자 뭉치와 가장 비슷한 문서 찾기
docs = vectorstore.similarity_search(question, k=3) # 상위 3개 검색

for doc in docs:
    print(f"찾은 내용: \n{doc.page_content[:50]}")

질문 벡터 샘플(5개): [-0.013053768, -0.0072450587, 0.023864685, -0.07714901, -0.027311333]
찾은 내용: 전문 요약 및 핵심 가치 (Professional Summary) 
 
성명: 이순신 (L
찾은 내용: 리더십, 교육 및 미래 비전 (Beyond Coding) 
커뮤니티 및 오픈소스 활동: '
찾은 내용: 프로젝트 사례 1 - 엔터프라이즈급 지식 베이스 RAG 구축 
 
프로젝트명: A사 내부 


In [ ]:

# 5. 프롬프트 템플릿 설정 (질문 형식을 정의)
template = """다음 문맥을 활용하여 질문에 답하세요:
{context}

질문: {question}
답변:"""
prompt = PromptTemplate.from_template(template)

# 6. 체인(Chain) 생성 (LCEL 방식: 부품들을 파이프로 연결)
chain = (
        {"context": retriever, "question": RunnablePassthrough()}
        | prompt
        | llm
        | StrOutputParser()
)

# 7. 실행
# ans = chain.invoke("이 문서에서 가장 중요한 내용은 뭐야?")
# print(ans)
for ans in chain.stream("10줄 이내로 요약해줘 핵심"):
    print(ans, end="", flush=True)



In [ ]:

# 5. 프롬프트 템플릿 개량
template = """당신은 유능한 비서입니다.
1. 아래 '문맥'에 질문과 관련된 내용이 있다면, 그 내용을 바탕으로 우선적으로 답변하세요.
2. 만약 '문맥'에 관련 내용이 전혀 없다면, 당신이 원래 알고 있는 지식을 활용하여 친절하게 답변하세요.
3. 문서 내용을 사용할 때는 "문서에 따르면~"이라고 언급하고, 일반 지식일 때는 "제 지식에 따르면~"이라고 구분해주면 더 좋습니다.

문맥:
{context}

질문: {question}
답변:"""

prompt = PromptTemplate.from_template(template)

# 6. 체인(Chain) 생성 (LCEL 방식: 부품들을 파이프로 연결)
chain = (
        {"context": retriever, "question": RunnablePassthrough()}
        | prompt
        | llm
        | StrOutputParser()
)

# 7. 실행
for ans in chain.stream("오늘 점심 메뉴 추천해줘"):
    print(ans, end="", flush=True)

In [48]:
from langchain_community.chat_message_histories import ChatMessageHistory

# 1. 메모리 저장소 생성 (자바의 ConcurrentHashMap 같은 역할)
history = ChatMessageHistory()

# 2. 프롬프트 수정 (대화 기록 {chat_history} 추가)
template = """당신은 유능한 비서입니다. 이전 대화 맥락과 제공된 문맥을 바탕으로 답하세요.
문서에 내용이 없다면 당신의 지식을 활용하되, 문서 내용일 때는 출처를 밝히세요.

이전 대화 기록:
{chat_history}

문맥:
{context}

질문: {question}
답변:"""
prompt = PromptTemplate.from_template(template)

# 3. 체인 재구성 (이전과 동일하지만 prompt에 history가 들어감)
chain = (
        {"context": retriever, "question": RunnablePassthrough(), "chat_history": lambda x: history.messages}
        | prompt
        | llm
        | StrOutputParser()
)

# 4. 연속 대화 테스트 함수
def ask_question(query):
    print(f"\nQ: {query}")
    print("A: ", end="")
    full_answer = ""
    for chunk in chain.stream(query):
        print(chunk, end="", flush=True)
        full_answer += chunk

    # 대화 기록 저장 (자바의 list.add() 느낌)
    history.add_user_message(query)
    history.add_ai_message(full_answer)
    print("\n")

# --- 테스트 시작 ---
ask_question("이준수가 누구야?")
ask_question("그 사람의 핵심 기술 스택 3가지만 뽑아줘.") # '그 사람'이 누군지 기억함!
ask_question("방금 말한 기술들 중에서 자바 개발자에게 왜 중요한지도 설명해줘.")



Q: 이준수가 누구야?
A: 문서 내용에 '이준수'라는 이름은 직접적으로 언급되어 있지 않습니다. 다만, 문서의 메타데이터에는 작성자(author)가 'junsu'로 표시되어 있습니다. (출처: Document metadata)

문서의 주요 내용은 '이순신'이라는 인물에 대한 프로필입니다. 이순신은 총 8년 경력(Java Backend 5년, AI/LLM Engineering 3년)의 개발자로, 가상화 기반의 AI 인프라 설계, RAG(Retrieval-Augmented Generation) 최적화, 고가용성 마이크로서비스 아키텍처(MSA) 구축을 전문으로 합니다. 그는 초기 Java Spring 프레임워크 기반의 엔터프라이즈 시스템 개발자로 커리어를 시작했으며, 최근 3년 동안은 생성형 AI 기술을 활용하여 기업용 데이터를 안전하고 정확하게 활용할 수 있는 RAG 아키텍처를 전문적으로 연구해왔습니다. (출처: Document e872da16-54a1-4af0-abd8-b45b083960f5, page 0)

또한 이순신은 'Java 개발자를 위한 AI 시작하기' 세미나를 진행하고 LangChain4j 한국어 문서화 작업에 참여하는 등 커뮤니티 및 오픈소스 활동에도 기여하고 있습니다. 그의 미래 비전은 인프라의 확장성(Scalability)과 AI의 지능(Intelligence)을 연결하는 하이브리드 엔지니어로 거듭나는 것입니다. (출처: Document 6393ae8c-6b45-4b48-8390-47b74f5f720a, page 4)


Q: 그 사람의 핵심 기술 스택 3가지만 뽑아줘.
A: 이순신의 핵심 기술 스택 3가지는 다음과 같습니다.

1.  **Java Backend & 프레임워크**: Spring Boot 3.x, Spring Cloud, Spring AI를 활용한 MSA 구축 및 JVM 튜닝, 가비지 컬렉션(GC) 최적화 경험을 보유하고 있습니다. (출처: Document e07a45b5-a327-4af3-b075-cf5958196971, 

In [ ]:
from langchain_core.runnables import RunnableParallel

rag_chain_with_source = RunnableParallel(
    {"context": retriever, "question": RunnablePassthrough()}
).assign(
    answer=(
            PromptTemplate.from_template(
                """아래 문맥을 사용하여 질문에 답하세요.
                문맥: {context}
                질문: {question}
                답변:"""
            )
            | llm
            | StrOutputParser()
    )
)

# 2. 실행 및 결과 확인
question = "이준수의 주요 프로젝트 2가지를 요약하고 각각 몇 페이지에 있는지 알려줘."
result = rag_chain_with_source.invoke(question)

# 3. 결과 출력
print(f"--- [AI 답변] ---\n{result['answer']}\n")

print("--- [참고한 문서 조각들] ---")
for i, doc in enumerate(result['context']):
    # doc.metadata 안에 파일명, 페이지 번호 등이 들어있습니다.
    page = doc.metadata.get('page', '알 수 없음')
    source = doc.metadata.get('source', '알 수 없음')
    print(f"[{i+1}] 출처: {source} (페이지: {page})")
    print(f"    내용 요약: {doc.page_content[:50]}...")